# 📊 Notebook 5: Model Evaluation

Confusion matrix, ROC curves, precision-recall, feature importance, and final model comparison.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score)
import warnings; warnings.filterwarnings('ignore')

# Rebuild dataset (same as notebook 4)
df = pd.read_csv('../data/processed/cleaned_churn_data.csv')
for col in ['Partner','Dependents','PhoneService','PaperlessBilling']:
    df[col] = df[col].map({'Yes':1,'No':0})
df['gender'] = df['gender'].map({'Male':1,'Female':0})
cats = ['InternetService','Contract','PaymentMethod','MultipleLines',
        'OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df_enc = pd.get_dummies(df, columns=cats).fillna(0)
X = df_enc.drop(columns=['Churn','customerID','TenureCohort'], errors='ignore')
y = df_enc['Churn']
scaler = StandardScaler()
X_sc = X.copy()
for c in ['tenure','MonthlyCharges','TotalCharges']:
    X_sc[c] = scaler.fit_transform(X[[c]])
X_train,X_test,y_train,y_test = train_test_split(X_sc,y,test_size=0.2,random_state=42,stratify=y)
models = {
    'Logistic Regression': LogisticRegression(solver='lbfgs',max_iter=1000,random_state=42),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100,random_state=42,n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100,random_state=42),
}
[m.fit(X_train,y_train) for m in models.values()]
print("All models trained ✓")

## 5.1 Confusion Matrix — Logistic Regression

In [ ]:
lr = models['Logistic Regression']
cm = confusion_matrix(y_test, lr.predict(X_test))
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Retained','Churned'], yticklabels=['Retained','Churned'])
plt.title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.ylabel('Actual'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()
tn,fp,fn,tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

## 5.2 ROC Curves — All Models

In [ ]:
colors = ['#2dd4bf','#86efac','#f59e0b','#a78bfa']
plt.figure(figsize=(8,6))
for (name, model), color in zip(models.items(), colors):
    proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    score = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={score:.3f})')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison', fontweight='bold')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

## 5.3 Precision-Recall Curve

In [ ]:
plt.figure(figsize=(8,6))
for (name, model), color in zip(models.items(), colors):
    proba = model.predict_proba(X_test)[:,1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    plt.plot(rec, prec, color=color, lw=2, label=name)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curves', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

## 5.4 Full Metrics Comparison

In [ ]:
rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    rows.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test,y_pred),
        'Precision': precision_score(y_test,y_pred,zero_division=0),
        'Recall':    recall_score(y_test,y_pred,zero_division=0),
        'F1':        f1_score(y_test,y_pred,zero_division=0),
        'AUC-ROC':   roc_auc_score(y_test,y_proba),
    })
results = pd.DataFrame(rows).set_index('Model')
print(results.round(4).to_string())

## 5.5 Feature Importance — Random Forest

In [ ]:
rf = models['Random Forest']
fi = pd.Series(rf.feature_importances_, index=X.columns).nlargest(15)
fi[::-1].plot.barh(figsize=(9,6), color='#2dd4bf')
plt.title('Top 15 Feature Importances (Random Forest)', fontweight='bold')
plt.xlabel('Importance Score'); plt.tight_layout(); plt.show()
print(fi.round(4).to_string())

## 5.6 Final Recommendation

In [ ]:
print("=" * 60)
print("BEST MODEL: Logistic Regression")
print("=" * 60)
print("  • AUC-ROC: 84.03%  (2nd overall, 0.46% behind GB)")
print("  • Recall:  54.81%  (HIGHEST — catches most churners)")
print("  • Fully interpretable coefficients for CRM integration")
print("  • Trains in <1s, scores customers in microseconds")
print()
print("Runner-up: Gradient Boosting (highest raw AUC-ROC 84.49%)")
print("  → Recommended when interpretability is less critical")